# Part IV exercises — the inverse problem

About 30 minutes. Exercise 4 is the most substantial and the most fun; start it early
if you are moving quickly.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_dmd as dmd
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (8, 3), "axes.grid": True, "grid.alpha": 0.3})

n_x, n_t = 10, 9
N_x, N_t = 2 ** n_x, 2 ** n_t
x = np.linspace(0, 1, N_x, endpoint=False)
dt = 0.004
t = np.arange(N_t) * dt

---
## Exercise 1 — read the temporal complexity off the bond

**Point:** the bond at the space/time interface counts the temporal degrees of freedom,
and you get it without running any algorithm.

Write down your prediction for each dataset below, then measure with
`tn.tt_ranks(dmd.space_time_qtt(data))[n_x]`.

1. $\sin(2\pi x)\cos(2\pi t)$
2. $\sin(2\pi(x - t))$
3. a sum of $m$ travelling waves with distinct speeds, for $m = 1, 2, 3, 4$
4. $\sin(2\pi x)\,\cos(2\pi t)$ plus noise of amplitude $10^{-3}$, at tolerances
   $10^{-10}$ and $10^{-2}$
5. a travelling *front*, $\tanh(20(x - 0.3 - 0.5t))$

In [ ]:
# TODO

Item 4 is the one to think hard about. What does the tolerance have to do with how
complicated the data *is*?

---
## Exercise 2 — break MPS-DMD on purpose

**Point:** DMD is a linear model in the observables you give it, and the failure is
structural, not numerical.

Apply `dmd.mps_dmd` to each of these and look at what comes back:

1. pure exponential growth, $e^{0.5t}\sin(2\pi x)$ — should be exact;
2. a **standing** wave, $\sin(2\pi x)\cos(2\pi t)$ — try rank 1 and rank 2 and explain
   what you see;
3. a frequency-swept wave, $\sin(2\pi(x - t(1 + 0.5t)))$;
4. a nonlinearity, $u(x,t) = \tanh\!\big(3\sin(2\pi x)\,e^{t}\big)$.

For each, report the eigenvalues and — most usefully — whether the *reconstruction
within* the data was any good.

In [ ]:
# TODO

Case 2 is the classic trap: a standing wave has spatial rank 1, so no DMD can find the
two eigenvalues $e^{\pm 2\pi i \Delta t}$ that generate it. The fix is a time-delay
(Hankel) embedding — which, in the space-time train, is a statement about which sites
you cut. Worth a discussion even if you do not implement it.

---
## Exercise 3 — truncation as a denoiser

**Point:** it often works, nothing tells you when, and that is an open problem rather
than a technique.

Add Gaussian noise of amplitude $\sigma$ to the three-mode dataset and sweep the
truncation rank. For each $(\sigma, \chi)$ record the error in the recovered
eigenvalues. Plot the error against $\chi$ for each $\sigma$ and find the optimum.

In [ ]:
c1, c2, decay = 1.0, -0.4, -0.3
clean = (np.sin(2 * np.pi * (x[:, None] - c1 * t[None, :]))
         + 0.6 * np.cos(6 * np.pi * (x[:, None] - c2 * t[None, :]))
         + 0.5 * np.exp(decay * t)[None, :] * np.cos(10 * np.pi * x)[:, None])
expected = np.sort_complex(np.array([
    np.exp(2j * np.pi * c1 * dt), np.exp(-2j * np.pi * c1 * dt),
    np.exp(6j * np.pi * c2 * dt), np.exp(-6j * np.pi * c2 * dt),
    np.exp(decay * dt)]))

# TODO

Then the question that matters: **in a real problem you do not have `expected`.** What
would you monitor instead, and how confident should you be?

---
## Exercise 4 — solve a PDE all at once, with no time loop

**Point:** if space and time are just sites on one chain, "time stepping" is only one
way to solve the system, and not obviously the best one.

Assemble the entire space-time system for the heat equation as a single MPO and solve
it with one DMRG call. Backward Euler on a space-time grid reads

$$\frac{U_k - U_{k-1}}{\Delta\tau} - \alpha L_x U_k = f_k,
  \qquad U_{-1} \equiv 0, \qquad f_0 = \frac{u_0}{\Delta\tau},$$

which is one linear system $M\,U = b$ over $n_x + n_t$ sites.

Steps:

1. build $D_t = (\mathbb{1} - S_t)/\Delta\tau$ on the time sites, with $S_t$ the
   non-periodic shift — its "missing" first column is exactly the $U_{-1}=0$ condition;
2. build $M = \mathbb{1}_x \otimes D_t - \alpha L_x \otimes \mathbb{1}_t$;
3. build $b = (u_0/\Delta\tau) \otimes \delta_{t,0}$, where $\delta_{t,0}$ is the rank-1
   train with cores $(1, 0)$;
4. $M$ is not symmetric and `dmrg_solve` wants an SPD operator, so solve the normal
   equations $M^\top M\,U = M^\top b$. Transposing an MPO is
   `c.transpose(0, 2, 1, 3)` on every core;
5. check the answer against the backward-Euler recursion you never ran.

In [ ]:
nx, nt = 8, 6
Nx, Nt = 2 ** nx, 2 ** nt
h = 2.0 ** -nx
alpha = 1.0
T = 0.002
dtau = T / Nt
v0 = np.sin(np.pi * (np.arange(Nx) + 1) / (Nx + 1))          # a discrete eigenvector
lam = -4.0 / h ** 2 * np.sin(np.pi / (2 * (Nx + 1))) ** 2     # its eigenvalue

# TODO

Once it works, the interesting part: swap $u_0$ for something non-separable in
space-time — a narrow Gaussian bump, say — and watch the space/time bond dimension.
That number is what an all-at-once method has to pay, and it is what decides whether
removing the time loop was a good idea.

---
## Solutions

In [ ]:
# --- Exercise 1 ---
rng = np.random.default_rng(0)
cases = {
    "standing sin(x)cos(t)": np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :],
    "travelling sin(x - t)": np.sin(2 * np.pi * (x[:, None] - t[None, :])),
    "2 travelling waves": sum(np.sin(2 * np.pi * (x[:, None] - c * t[None, :]))
                              for c in (1.0, -0.4)),
    "3 travelling waves": sum(np.sin(2 * np.pi * (x[:, None] - c * t[None, :]))
                              for c in (1.0, -0.4, 2.2)),
    "4 travelling waves": sum(np.sin(2 * np.pi * (x[:, None] - c * t[None, :]))
                              for c in (1.0, -0.4, 2.2, 0.7)),
    "travelling front (tanh)": np.tanh(20 * (x[:, None] - 0.3 - 0.5 * t[None, :])),
}
for name, data in cases.items():
    c = dmd.space_time_qtt(data, eps=1e-10)
    print(f"{name:>26}: space/time bond = {tn.tt_ranks(c)[n_x]:>4}")

noisy = (np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :]
         + 1e-3 * rng.standard_normal((N_x, N_t)))
for eps in (1e-10, 1e-2):
    c = dmd.space_time_qtt(noisy, eps=eps)
    print(f"{'standing + 1e-3 noise':>26}: space/time bond = "
          f"{tn.tt_ranks(c)[n_x]:>4}  (tolerance {eps:.0e})")
print("""
Each travelling wave contributes 2 to the bond (it is a sum of two products), a standing
wave contributes 1, and the tanh front is not a finite sum of products at all -- its bond
is set by how accurately you insist on representing it. The noise case makes the point
sharply: at 1e-10 the bond saturates, because you have demanded that the noise be
reproduced exactly; at 1e-2 it collapses back to 1. The bond measures complexity *at the
accuracy you asked for*, which is the only kind of complexity there is.""")

In [ ]:
# --- Exercise 2 ---
def try_dmd(name, data, rank):
    c = dmd.space_time_qtt(data, eps=1e-10)
    r = dmd.mps_dmd(c, n_x, rank=rank)
    recon = np.array([dmd.predict_dense(r, k) for k in range(0, N_t, N_t // 8)]).T
    ref = data[:, ::N_t // 8]
    err_in = np.linalg.norm(recon - ref) / np.linalg.norm(ref)
    print(f"{name:>24} rank {rank}: |lambda| = "
          f"{np.round(np.sort(np.abs(r['eigenvalues'])), 5)},  "
          f"in-sample error {err_in:.2e}")


try_dmd("exponential growth",
        np.exp(0.5 * t)[None, :] * np.sin(2 * np.pi * x)[:, None], 1)
try_dmd("standing wave",
        np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :], 1)
try_dmd("standing wave",
        np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :], 2)
try_dmd("swept frequency",
        np.sin(2 * np.pi * (x[:, None] - t[None, :] * (1 + 0.5 * t[None, :]))), 6)
try_dmd("tanh nonlinearity",
        np.tanh(3 * np.sin(2 * np.pi * x)[:, None] * np.exp(t)[None, :]), 6)
print("""
The exponential is exact: it genuinely is a one-mode linear system. The standing wave has
spatial rank 1, so DMD can only ever return ONE eigenvalue, and asking for rank 2 does not
help -- the two frequencies that generate cos(2 pi t) are simply not visible in a
one-dimensional column space. The swept and tanh cases fit the in-sample data tolerably
and mean nothing outside it, because the dynamics are not linear in the observables
supplied. Note that DMD does not fail loudly in any of these. You have to check.""")

In [ ]:
# --- Exercise 3 ---
plt.figure(figsize=(8, 3.5))
for sigma in (0.01, 0.05, 0.2):
    noisy = clean + sigma * rng.standard_normal(clean.shape)
    chis, errs = [], []
    for chi in (3, 5, 8, 12, 20, 40, 80):
        c = dmd.space_time_qtt(noisy, eps=1e-12, chi_max=chi)
        r = dmd.mps_dmd(c, n_x, rank=5)
        lam_got = np.sort_complex(r["eigenvalues"])
        if len(lam_got) < 5:
            lam_got = np.pad(lam_got, (0, 5 - len(lam_got)))
        chis.append(chi)
        errs.append(np.abs(lam_got - expected).max())
    plt.loglog(chis, errs, "o-", label=f"$\\sigma={sigma}$")
    print(f"sigma = {sigma:>4}: best chi = {chis[int(np.argmin(errs))]:>3}, "
          f"eigenvalue error {min(errs):.2e}")
plt.xlabel("truncation rank $\\chi$"), plt.ylabel("max eigenvalue error")
plt.legend(), plt.tight_layout()
print("""
Read the optimum as a function of noise. At sigma = 0.01 the error keeps falling as chi
grows: the noise is small enough that keeping a little of it costs less than discarding
signal. At sigma = 0.05 and above the optimum drops back to about the true rank, and
over-truncating and under-truncating both hurt. In a real problem you have no
`expected` to plot against. What you can monitor is the singular-value spectrum at the
space/time bond, looking for a knee; the stability of the eigenvalues as chi varies; and
whether the modes are physically plausible. None of these is a guarantee, and error bars
on tensor-network predictions are essentially an open problem.""")

In [ ]:
# --- Exercise 4 ---
def eye_mpo(k):
    return [np.eye(2).reshape(1, 2, 2, 1) for _ in range(k)]


Lx = tn.qtt_laplacian(nx, dx=h)
St = tn.qtt_shift(nt, +1)                                   # (S u)[k] = u[k-1]
Dt = tn.mpo_scale(
    tn.mpo_round(tn.mpo_add(tn.mpo_identity(nt), tn.mpo_scale(St, -1.0)), 1e-13),
    1.0 / dtau)

M = tn.mpo_round(tn.mpo_add(eye_mpo(nx) + Dt,
                            tn.mpo_scale(Lx + eye_mpo(nt), -alpha)), 1e-12)
MT = [c.transpose(0, 2, 1, 3).copy() for c in M]
A = tn.mpo_round(tn.mpo_compose(MT, M), 1e-12)

delta0 = [np.array([1.0, 0.0]).reshape(1, 2, 1) for _ in range(nt)]
b = tn.tt_round(tn.qtt_from_vector(v0 / dtau, eps=1e-13) + delta0, 1e-13)
rhs = tn.tt_round(tn.mpo_apply(MT, b), 1e-12)

t0 = time.perf_counter()
U = tn.dmrg_solve(A, rhs, sweeps=6, eps=1e-10, chi_max=40)
print(f"M rank {max(tn.mpo_ranks(M))}, M^T M rank {max(tn.mpo_ranks(A))}")
print(f"one solve for ALL {Nt} timesteps: {time.perf_counter() - t0:.2f} s, "
      f"chi = {max(tn.tt_ranks(U))}, residual {tn.residual(A, U, rhs):.1e}")

full = tn.tt_full(U).reshape(Nx, Nt)
for k in (0, Nt // 2, Nt - 1):
    be = v0 / (1 - lam * dtau) ** (k + 1)                   # backward Euler in closed form
    print(f"  t index {k:>2}: error vs backward Euler "
          f"{np.linalg.norm(full[:, k] - be) / np.linalg.norm(be):.2e}")

In [ ]:
# --- Exercise 4, the interesting variant: a non-separable initial condition ---
xg = np.linspace(0, 1, Nx, endpoint=False)
for label, u0 in [("eigenvector (separable in space-time)", v0),
                  ("narrow gaussian bump", np.exp(-((xg - 0.5) / 0.02) ** 2))]:
    b2 = tn.tt_round(tn.qtt_from_vector(u0 / dtau, eps=1e-13) + delta0, 1e-13)
    rhs2 = tn.tt_round(tn.mpo_apply(MT, b2), 1e-12)
    U2 = tn.dmrg_solve(A, rhs2, sweeps=6, eps=1e-10, chi_max=60)
    print(f"{label:>38}: space/time bond = {tn.tt_ranks(U2)[nx]:>3}, "
          f"max chi = {max(tn.tt_ranks(U2)):>3}")
print("""
The separable case costs a bond of 1 or 2; the Gaussian costs more, because the diffusion
of a localised bump is not a short sum of products of a function of x and a function of t.
That bond is the price of removing the time loop, and it is exactly what the space-time
papers spend their effort bounding.""")